In [1]:
import jax
import jax.numpy as jnp

global_list = []

def log2(x):
  global_list.append(x)
  ln_x = jnp.log(x)
  ln_2 = jnp.log(2.0)
  return ln_x / ln_2

print(jax.make_jaxpr(log2)(3.0))

{ lambda ; a:f32[]. let
    b:f32[] = log a
    c:f32[] = log 2.0:f32[]
    d:f32[] = div b c
  in (d,) }


In [3]:
def log2_with_print(x):
  print("printed x:", x)
  ln_x = jnp.log(x)
  ln_2 = jnp.log(2.0)
  return ln_x / ln_2

print(jax.make_jaxpr(log2_with_print)(3.))

printed x: JitTracer(~float32[])
{ lambda ; a:f32[]. let
    b:f32[] = log a
    c:f32[] = log 2.0:f32[]
    d:f32[] = div b c
  in (d,) }


In [6]:
def log2_if_rank_2(x):
  if x.ndim == 2:
    ln_x = jnp.log(x)
    ln_2 = jnp.log(2.0)
    return ln_x / ln_2
  else:
    return x

print(jax.make_jaxpr(log2_if_rank_2)(jax.numpy.array([1, 2, 3])))
print(jax.make_jaxpr(log2_if_rank_2)(jax.numpy.array([1, 3])))

{ lambda ; a:i32[3]. let  in (a,) }
{ lambda ; a:i32[2]. let  in (a,) }


In [7]:
import jax
import jax.numpy as jnp

def selu(x, alpha=1.67, lambda_=1.05):
  return lambda_ * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

x = jnp.arange(1000000)
%timeit selu(x).block_until_ready()

1.51 ms ± 509 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
selu_jit = jax.jit(selu)

# Pre-compile the function before timing...
selu_jit(x).block_until_ready()

%timeit selu_jit(x).block_until_ready()

1.28 ms ± 10.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
# Condition on value of x.

def f(x):
  if x > 0:
    return x
  else:
    return 2 * x

(f)(10)  # Raises an error

10

In [14]:
# While loop conditioned on x and n.

def g(x, n):
  i = 0
  while i < n:
    i += 1
  return x + i

%timeit (g)(10, 20)

335 ns ± 1.84 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [15]:
# While loop conditioned on x and n with a jitted body.

@jax.jit
def loop_body(prev_i):
  return prev_i + 1

def g_inner_jitted(x, n):
  i = 0
  while i < n:
    i = loop_body(i)
  return x + i

%timeit g_inner_jitted(10, 20)

40.3 ms ± 1.71 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
f_jit_correct = jax.jit(f, static_argnums=0)
print(f_jit_correct(10))

10


In [18]:
g_jit_correct = jax.jit(g, static_argnames=['n'])
print(g_jit_correct(10, 20))

30


In [19]:
from functools import partial

@jax.jit(static_argnames=['n'])
def g_jit_decorated(x, n):
  i = 0
  while i < n:
    i += 1
  return x + i

print(g_jit_decorated(10, 20))

30


In [20]:
from functools import partial

def unjitted_loop_body(prev_i):
  return prev_i + 1

def g_inner_jitted_partial(x, n):
  i = 0
  while i < n:
    # Don't do this! each time the partial returns
    # a function with different hash
    i = jax.jit(partial(unjitted_loop_body))(i)
  return x + i

def g_inner_jitted_lambda(x, n):
  i = 0
  while i < n:
    # Don't do this!, lambda will also return
    # a function with a different hash
    i = jax.jit(lambda x: unjitted_loop_body(x))(i)
  return x + i

def g_inner_jitted_normal(x, n):
  i = 0
  while i < n:
    # this is OK, since JAX can find the
    # cached, compiled function
    i = jax.jit(unjitted_loop_body)(i)
  return x + i

print("jit called in a loop with partials:")
%timeit g_inner_jitted_partial(10, 20).block_until_ready()

print("jit called in a loop with lambdas:")
%timeit g_inner_jitted_lambda(10, 20).block_until_ready()

print("jit called in a loop with caching:")
%timeit g_inner_jitted_normal(10, 20).block_until_ready()

jit called in a loop with partials:
676 ms ± 7.56 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
jit called in a loop with lambdas:
673 ms ± 3.74 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
jit called in a loop with caching:
45.2 ms ± 1.21 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [21]:
import jax
import jax.numpy as jnp

x = jnp.arange(5)
w = jnp.array([2., 3., 4.])

def convolve(x, w):
  output = []
  for i in range(1, len(x)-1):
    output.append(jnp.dot(x[i-1:i+2], w))
  return jnp.array(output)

convolve(x, w)

Array([11., 20., 29.], dtype=float32)

In [22]:
xs = jnp.stack([x, x])
ws = jnp.stack([w, w])

In [28]:
def manually_batched_convolve(xs, ws):
  output = []
  for i in range(xs.shape[0]):
    output.append(convolve(xs[i], ws[i]))
  return jnp.stack(output)

%timeit manually_batched_convolve(xs, ws)

5.35 ms ± 73.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [27]:
def manually_vectorized_convolve(xs, ws):
  output = []
  for i in range(1, xs.shape[-1] -1):
    output.append(jnp.sum(xs[:, i-1:i+2] * ws, axis=1))
  return jnp.stack(output, axis=1)

%timeit manually_vectorized_convolve(xs, ws)

1.85 ms ± 22.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [26]:
auto_batch_convolve = jax.vmap(convolve)

%timeit auto_batch_convolve(xs, ws)

3.18 ms ± 109 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [29]:
auto_batch_convolve_v2 = jax.vmap(convolve, in_axes=1, out_axes=1)

xst = jnp.transpose(xs)
wst = jnp.transpose(ws)

auto_batch_convolve_v2(xst, wst)

Array([[11., 11.],
       [20., 20.],
       [29., 29.]], dtype=float32)

In [30]:
batch_convolve_v3 = jax.vmap(convolve, in_axes=[0, None])

batch_convolve_v3(xs, w)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

In [32]:
jitted_batch_convolve = jax.jit(auto_batch_convolve)

%timeit jitted_batch_convolve(xs, ws)

66.6 μs ± 783 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [33]:
def pairwise(f, xs):
  return jax.vmap(lambda x: jax.vmap(lambda y: f(x, y))(xs))(xs)